## Step 1: Install Required Library


In [1]:
%pip install pageindex
%pip install langchain langchain-google-genai google-generativeai

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chromadb 1.5.5 requires numpy>=1.22.5, which is not installed.
dagshub-annotation-converter 0.1.13 requires pandas, which is not installed.
gradio 5.20.0 requires numpy<3.0,>=1.0, which is not installed.
gradio 5.20.0 requires pandas<3.0,>=1.0, which is not installed.
langchain-chroma 1.1.0 requires numpy>=1.26.0; python_version < "3.13", which is not installed.
manim 0.19.1 requires numpy>=2.0, which is not installed.
manim 0.19.1 requires numpy>=2.1; python_full_version >= "3.10", which is not installed.
mlflow 3.3.2 requires cryptography<46,>=43.0.0, which is not installed.
mlflow 3.3.2 requires numpy<3, which is not installed.
mlflow 3.3.2 requires pandas<3, which is not installed.
nibabel 5.3.2 requires numpy>=1.22, which is not installed.
nipype 1.9.2 requires numpy>=1.21, which is not installed.
plotly-resa

  Using cached langchain_core-0.3.83-py3-none-any.whl.metadata (3.2 kB)
  Using cached numpy-2.4.4-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
Using cached langchain_core-0.3.83-py3-none-any.whl (458 kB)
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.3 MB 1.5 MB/s eta 0:00:08
   -- ------------------------------------- 0.8/12.3 MB 1.8 MB/s eta 0:00:07
   --- ------------------------------------ 1.0/12.3 MB 1.5 MB/s eta 0:00:08
   ---- ----------------------------------- 1.3/12.3 MB 1.4 MB/s eta 0:00:08
   ----- ---------------------------------- 1.6/12.3 MB 1.4 MB/s eta 0:00:08
   ----- ---------------------------------- 1.8/12.3 MB 1.4 MB/s eta 0:00:08
   ------- -------------------------------- 2.4/12.3 MB 1.6 MB/s eta 0:00:07
   -------- ------------------------------- 2.6

ERROR: Could not install packages due to an OSError: [Errno 28] No space left on device


[notice] A new release of pip is available: 25.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Import Required Libraries


In [9]:
import os
import json
import requests
import asyncio
from dotenv import load_dotenv

from pageindex import PageIndexClient
import pageindex.utils as utils
load_dotenv()


True

## Step 3: Initialize PageIndex Client


In [10]:
PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)


## Step 4: Download the PDF Document


In [11]:
pdf_url = "https://arxiv.org/pdf/2501.12948.pdf"
pdf_path = os.path.join("../data", pdf_url.split('/')[-1])

os.makedirs(os.path.dirname(pdf_path), exist_ok=True)

response = requests.get(pdf_url)
with open(pdf_path, "wb") as f:
    f.write(response.content)

print(f"Downloaded {pdf_url}")

Downloaded https://arxiv.org/pdf/2501.12948.pdf


## Step 5: Submit Document to PageIndex


In [12]:
doc_info = pi_client.submit_document(pdf_path)
doc_id = doc_info["doc_id"]

print('Document Submitted:', doc_id)

Document Submitted: pi-cmnmy1nmo04qb01qrzrwvrn3e


## Step 6: Indexing
Since indexing takes time, we poll the system until the document is ready.

Use is_retrieval_ready() to check indexing status.
Implement retry logic with timeout handling.
Once ready, fetch and print the document tree.

In [13]:
import time

print(f"Waiting for document {doc_id} to be indexed...")

max_retries = 30 
retry_count = 0

while not pi_client.is_retrieval_ready(doc_id):
    if retry_count >= max_retries:
        print("Timeout: Document processing took too long.")
        break
    
    print(f"Still processing... (Attempt {retry_count + 1}/{max_retries})")
    time.sleep(5)
    retry_count += 1

if pi_client.is_retrieval_ready(doc_id):
    print("Success! Document is ready.")
    tree = pi_client.get_tree(doc_id, node_summary=True)['result']
    utils.print_tree(tree)
else:
    tree = None

Waiting for document pi-cmnmy1nmo04qb01qrzrwvrn3e to be indexed...
Still processing... (Attempt 1/30)
Still processing... (Attempt 2/30)
Still processing... (Attempt 3/30)
Still processing... (Attempt 4/30)
Still processing... (Attempt 5/30)
Still processing... (Attempt 6/30)
Still processing... (Attempt 7/30)
Still processing... (Attempt 8/30)
Still processing... (Attempt 9/30)
Still processing... (Attempt 10/30)
Still processing... (Attempt 11/30)
Still processing... (Attempt 12/30)
Still processing... (Attempt 13/30)
Still processing... (Attempt 14/30)
Still processing... (Attempt 15/30)
Still processing... (Attempt 16/30)
Still processing... (Attempt 17/30)
Still processing... (Attempt 18/30)
Still processing... (Attempt 19/30)
Still processing... (Attempt 20/30)
Still processing... (Attempt 21/30)
Still processing... (Attempt 22/30)
Still processing... (Attempt 23/30)
Still processing... (Attempt 24/30)
Still processing... (Attempt 25/30)
Still processing... (Attempt 26/30)
Still 

## Step 7: Initialize the LLM
Next, we configure the Large Language Model that will generate answers using retrieved context.




In [ ]:
from dotenv import load_dotenv
import os

load_dotenv() 

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
    api_key=GOOGLE_API_KEY
)

## Step 8: Define Retrieval Function (Vectorless Retrieval)
This function sends the user’s question to PageIndex, waits until the system finds the most relevant sections in the document and then extracts the useful text content for answer generation.

First, it submits the query to PageIndex using submit_query() and receives a retrieval_id.
Then, it continuously checks (polls) the retrieval status until it is completed.
Finally, it collects the relevant text content from the top matching nodes and returns them as context.

In [17]:
def retrieve_from_pageindex(query, doc_id, top_k=3):
    
    response = pi_client.submit_query(
        doc_id=doc_id,
        query=query
    )

    retrieval_id = response.get("retrieval_id")
    if not retrieval_id:
        return []

    while True:
        retrieval = pi_client.get_retrieval(retrieval_id)
        status = retrieval.get("status")

        if status == "completed":
            break
        elif status == "failed":
            return []

        time.sleep(1)

    nodes = retrieval.get("retrieved_nodes", [])
    contexts = []

    for node in nodes[:top_k]:
        relevant_contents = node.get("relevant_contents", [])
        
        for group in relevant_contents:
            for item in group:
                content = item.get("relevant_content")
                if content:
                    contexts.append(content)

    return contexts

## Step 9: Build Vectorless RAG Pipeline
This function combines retrieved context and sends it to the LLM.

Retrieve structured content from PageIndex.
Combine context into a single prompt.
Ask the LLM to answer strictly from retrieved context.

In [18]:
def vectorless_rag(query, doc_id):

    contexts = retrieve_from_pageindex(query, doc_id)

    if not contexts:
        return "No relevant context found."

    combined_context = "\n\n".join(contexts)

    prompt = f"""
    You are a research assistant.
    Answer ONLY using the context below.
    If the answer is not found, say "Not found in document."

    Context:
    {combined_context}

    Question:
    {query}
    """
    response = llm.invoke(prompt)
    return response.content

## Step 10: Run Query and Generate Final Answer
Here we provide a question and generate the answer using Vectorless RAG.




In [19]:
query = "What is the main contribution of this paper?"

answer = vectorless_rag(query, doc_id)

print("\nFINAL ANSWER:\n")
print(answer)


FINAL ANSWER:

The main contribution of the paper is exploring the development of reasoning abilities in large language models (LLMs) through self-evolution in a reinforcement learning (RL) framework with minimal human labeling.

Additionally, the main contribution of the paper is:
*   The development of DeepSeek-R1-Zero, a large language model trained with reinforcement learning to autonomously enhance its reasoning capabilities.
*   The development of DeepSeek-R1, a model designed to overcome the limitations of DeepSeek-R1-Zero, such as poor readability and language mixing, by employing a multi-stage training pipeline.
*   The development of DeepSeek-R1, which integrates multiple reward signals—reasoning rewards, general rewards, and language consistency rewards—to align the model with human preferences and improve readability.
